# Fractional ODE Solver via fPINN

This notebook trains a Physics-Informed Neural Network (PINN) to solve a
fractional ordinary differential equation (fODE) of the form:

$$\frac{\partial^{\alpha} u(t)}{\partial t^{\alpha}} = -u(t) + t^2 + \frac{8}{3\sqrt{\pi}} t^{3/2}, \quad 0 \le t \le 1$$

with initial condition $u(0) = 0$ and fractional order $\alpha = 0.5$.
The analytical solution is $u(t) = t^2$, used here as the reference for
benchmarking.

The Caputo fractional derivative is discretized using the **Diethelm** scheme.

## Contents

1. **Configuration** — all hyperparameters in one place.
2. **Imports & reproducibility** — library imports and random seed setup.
3. **Collocation points** — grid construction for training.
4. **Caputo derivative** — Diethelm discretization of the fractional derivative.
5. **Neural network model** — PINN architecture.
6. **PINN solver** — loss function, training loop.
7. **Training** — instantiate the solver and run it.
8. **Results** — comparison of fPINN prediction against the analytical solution.

## Notes

- Full-batch training (no minibatching).
- Adam optimizer with piecewise constant learning rate schedule.
- This notebook performs both training and plotting (no separate plotter).

## 1. Configuration

All tunable parameters live here. Change these — not the cells below.

In [ ]:
# ---- Domain ------------------------------------------------------------
TMIN, TMAX = 0.0, 1.0       # temporal domain

# ---- Collocation grid --------------------------------------------------
N_R = 30                    # number of residual collocation points
N_D = 30                    # number of initial-condition data points

# ---- Fractional derivative --------------------------------------------
ALPHA = 0.5                 # fractional order (Caputo)

# ---- Initial condition ------------------------------------------------
Y0 = 0.0                    # u(0) = 0

# ---- Network architecture ---------------------------------------------
NUM_HIDDEN_LAYERS = 3
NUM_NEURONS_PER_LAYER = 10
ACTIVATION = 'tanh'
KERNEL_INITIALIZER = 'glorot_normal'

# ---- Optimizer --------------------------------------------------------
# Piecewise constant learning rate schedule
LR_BOUNDARIES = [200, 1000]
LR_VALUES = [1e-2, 1e-3, 5e-4]

# ---- Training loop ----------------------------------------------------
MAX_ITERATIONS = 50001
CALLBACK_EVERY = 5000       # print loss every N iterations

# ---- Reproducibility --------------------------------------------------
RANDOM_SEED = 42

# ---- Data type --------------------------------------------------------
DTYPE = 'float32'

## 2. Imports and Reproducibility

In [ ]:
import sys

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Set default dtype for all TF operations
tf.keras.backend.set_floatx(DTYPE)

# Fix random seed for reproducibility
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 3. Collocation Points

We build two sets of points:
- **Residual collocation** `X_r` — uniform grid on $[t_{\min}, t_{\max}]$ with `N_R` points.
- **Initial condition** `X_data` — points at $t = t_{\min}$ used to enforce $u(0) = y_0$.

In [ ]:
# ---- Domain bounds ----------------------------------------------------
lb = tf.constant([TMIN], dtype=DTYPE)
ub = tf.constant([TMAX], dtype=DTYPE)

# ---- Initial condition points (all at t = TMIN) -----------------------
t_0 = tf.reshape(np.linspace(lb[0], lb[0], N_D, dtype=DTYPE), (-1, 1))
y_0 = tf.reshape(np.ones(N_D, dtype=DTYPE), (-1, 1))

X_data = t_0
u_data = y_0

# ---- Residual collocation grid (uniform in time) ----------------------
t_r = tf.reshape(np.linspace(lb[0], ub[0], N_R, dtype=DTYPE), (-1, 1))
X_r = tf.concat([t_r], axis=1)

# Grid spacing (used by the Caputo discretization)
h = (max(t_r) - min(t_r)) / (N_R - 1)

## 4. Caputo Fractional Derivative

Diethelm-type discretization of the Caputo fractional derivative. This operator
consumes a sequence of function values on the time grid and returns the
fractional derivative at a single time index.

In [ ]:
def gamma_tf(x):
    """Gamma function for TF tensors (via log-gamma)."""
    return tf.exp(tf.math.lgamma(x))


def caputo_derivative(alpha, f, data_point, h):
    """Diethelm-type discretization of the Caputo fractional derivative.

    Parameters
    ----------
    alpha : float
        Fractional order in (0, 1).
    f : tf.Tensor
        Function values on the time grid.
    data_point : int
        Time index at which to evaluate the derivative.
    h : float
        Time step.
    """
    summation = 0.0
    for k in range(data_point + 1):
        if k == 0:
            sigma = 1.0
        elif k == data_point:
            sigma = ((k - 1.0) ** (1.0 - alpha)
                     - k ** (1.0 - alpha)
                     + (1.0 - alpha) * k ** (-alpha))
        else:  # 0 < k < data_point
            sigma = ((k - 1.0) ** (1.0 - alpha)
                     - 2.0 * k ** (1.0 - alpha)
                     + (k + 1) ** (1.0 - alpha))
        summation += sigma * f[data_point - k]
    return h ** (-alpha) / gamma_tf(2.0 - alpha) * summation

## 5. Neural Network Model

In [ ]:
class PINN_NeuralNet(tf.keras.Model):
    """Fully-connected PINN architecture.

    Input: t (single scalar).
    Output: approximation of u(t).
    """

    def __init__(self, lb, ub,
                 output_dim=1,
                 num_hidden_layers=NUM_HIDDEN_LAYERS,
                 num_neurons_per_layer=NUM_NEURONS_PER_LAYER,
                 activation=ACTIVATION,
                 kernel_initializer=KERNEL_INITIALIZER,
                 **kwargs):
        super().__init__(**kwargs)

        self.num_hidden_layers = num_hidden_layers
        self.output_dim = output_dim
        self.lb = lb
        self.ub = ub

        self.hidden = [
            tf.keras.layers.Dense(
                num_neurons_per_layer,
                activation=tf.keras.activations.get(activation),
                kernel_initializer=kernel_initializer,
            )
            for _ in range(num_hidden_layers)
        ]
        self.out = tf.keras.layers.Dense(output_dim)

    def call(self, X):
        """Forward pass."""
        Z = X
        for layer in self.hidden:
            Z = layer(Z)
        return self.out(Z)

## 6. PINN Solver

The solver handles:
- **Residual assembly** via `get_r`, combining the fractional derivative and the forcing term.
- **Loss function** as a sum of equation and initial-condition losses.
- **Training loop** `solve_with_TFoptimizer`.

In [ ]:
class PINNSolver:
    """PINN solver for the fODE."""

    def __init__(self, model, X_r):
        self.model = model
        self.t = X_r

        # Training state
        self.hist = []
        self.iter = 0
        self.current_loss = None

    # ------------------------------------------------------------------
    # PDE residual
    # ------------------------------------------------------------------
    def get_r(self):
        """Assemble the ODE residual over the collocation grid."""
        pred = self.model(self.t)
        frac = [caputo_derivative(ALPHA, pred, dp, h) for dp in range(N_R)]

        # Governing ODE: D^alpha u = -u + t^2 + 8/(3*sqrt(pi)) * t^{3/2}
        rhs = -pred + self.t ** 2 + 8.0 * self.t ** 1.5 / 3.0 / tf.sqrt(np.pi)
        res = frac - rhs
        return res

    def loss_fn(self, X, u):
        """Total loss: residual + initial condition."""
        r = self.get_r()
        loss_eq = tf.reduce_mean(tf.square(r))

        y0_pred = self.model(X)
        loss_init = tf.reduce_mean(tf.square(y0_pred - Y0))

        return loss_eq + loss_init

    def get_grad(self, X, u):
        with tf.GradientTape(persistent=True) as tape:
            tape.watch(self.model.trainable_variables)
            loss = self.loss_fn(X, u)
        g = tape.gradient(loss, self.model.trainable_variables)
        del tape
        return loss, g

    # ------------------------------------------------------------------
    # Training loop
    # ------------------------------------------------------------------
    def solve_with_TFoptimizer(self, optimizer, X, u, N=MAX_ITERATIONS):
        """Train the network for at most N iterations."""
        @tf.function
        def train_step():
            loss, grad_theta = self.get_grad(X, u)
            optimizer.apply_gradients(zip(grad_theta, self.model.trainable_variables))
            return loss

        for i in range(N):
            loss = train_step()
            self.current_loss = loss.numpy()
            self.callback()

    # ------------------------------------------------------------------
    # Logging callback
    # ------------------------------------------------------------------
    def callback(self):
        if self.iter % CALLBACK_EVERY == 0:
            tf.print('It {:05d}: loss = {:10.4e}'.format(
                self.iter, self.current_loss), output_stream=sys.stdout)
        self.hist.append(self.current_loss)
        self.iter += 1

## 7. Training

In [ ]:
# ---- Instantiate model and solver -------------------------------------
model = PINN_NeuralNet(lb, ub)
model.build(input_shape=(None, 1))
solver = PINNSolver(model, X_r)

# ---- Build optimizer with piecewise constant learning rate ------------
lr = tf.keras.optimizers.schedules.PiecewiseConstantDecay(
    LR_BOUNDARIES, LR_VALUES
)
optim = tf.keras.optimizers.Adam(learning_rate=lr)

# ---- Run training -----------------------------------------------------
solver.solve_with_TFoptimizer(optim, X_data, u_data, N=MAX_ITERATIONS)

## 8. Results

Compare the fPINN prediction against the analytical solution $u(t) = t^2$.

In [ ]:
# ---- Matplotlib style -------------------------------------------------
from matplotlib import rc

rc('font', **{'family': 'Times'})
rc('text', usetex=True)
plt.rc('font', size=17)
plt.rc('axes', titlesize=20)
plt.rc('axes', labelsize=20)
plt.rc('xtick', labelsize=15)
plt.rc('ytick', labelsize=15)
plt.rc('legend', fontsize=16)
plt.rc('figure', titlesize=20)

COLORS = ['tab:blue', 'tab:orange']
MARKERS = ['o']
LINE_WIDTH = 3

In [ ]:
# ---- Evaluate model on a plotting grid --------------------------------
n_plot = 20
t_plot = np.reshape(np.linspace(lb[0], ub[0], n_plot), (-1, 1))
pred = model(t_plot)
y_exact = t_plot ** 2

# ---- Plot -------------------------------------------------------------
plt.figure(figsize=(5, 5))
plt.plot(t_plot, y_exact, lw=LINE_WIDTH, color=COLORS[0], label='$Ground\ truth$')
plt.scatter(t_plot, pred, marker=MARKERS[0], color=COLORS[1], label='$fPINN\ solution$')
plt.legend(frameon=False)
plt.ylabel('$u(t)$')
plt.xlabel('$t$')
plt.savefig('fODE_solution.png', dpi=300, transparent=False, bbox_inches='tight')
plt.show()